In [3]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from hbn.models.topic_modeling import *


## Topic Modeling with BERT
#### Leveraging BERT and TF-IDF to create easily interpretable topics.

See https://towardsdatascience.com/topic-modeling-with-bert-779f7db187e6 for a great tutorial (code in this notebook comes from the tutorial)

In [2]:
# load data
# dataframe contains sentences for all HBN clinical questionnaires

data_dir = '/Users/maedbhking/Documents/healthy_brain_network/data/raw/phenotype'
fname = 'item-names-cleaned.csv'

df = pd.read_csv(os.path.join(data_dir, fname))

# get sentences to be input to topic modeling routine
data = df['sentences']

## Embeddings

The very first step we have to do is converting the documents to numerical data. We use BERT for this purpose as it extracts different embeddings based on the context of the word. Not only that, there are many pre-trained models available ready to be used.

How you generate the BERT embeddings for a document is up to you. However, I prefer to use the `sentence-transformers` package as the resulting embeddings have shown to be of high quality and typically work quite well for document-level embeddings.

We are using Distilbert as it gives a nice balance between speed and performance. The package has several multi-lingual models available for you to use.

NOTE: Since transformer models have a token limit, you might run into some errors when inputting large documents. In that case, you could consider splitting documents into paragraphs.

In [ ]:
# get embeddings
embeddings = get_embeddings(data, transformer='distilbert-base-nli-mean-tokens')

## Clustering

We want to make sure that documents with similar topics are clustered together such that we can find the topics within these clusters. Before doing so, we first need to lower the dimensionality of the embeddings as many clustering algorithms handle high dimensionality poorly.

### UMAP
Out of the few dimensionality reduction algorithms, UMAP is arguably the best performing as it keeps a significant portion of the high-dimensional local structure in lower dimensionality.

We use the package `umap-learn` before we lower the dimensionality of the document embeddings. We reduce the dimensionality to 5 while keeping the size of the local neighborhood at 15. You can play around with these values to optimize for your topic creation. Note that a too low dimensionality results in a loss of information while a too high dimensionality results in poorer clustering results.

In [ ]:
# get umap embeddings
umap_embeddings = dimensionality_reduction(embeddings)

### HDBSAN
After having reduced the dimensionality of the documents embeddings to 5, we can cluster the documents with the `hdbscan` package. HDBSCAN is a density-based algorithm that works quite well with UMAP since UMAP maintains a lot of local structure even in lower-dimensional space. Moreover, HDBSCAN does not force data points to clusters as it considers them outliers.

In [ ]:
# get clusters
cluster = clustering(umap_embeddings)

Great! We now have clustered similar documents together which should represent the topics that they consist of. To visualize the resulting clusters we can further reduce the dimensionality to 2 and visualize the outliers as grey points:

In [ ]:
# visualize clusters
visualize_clusters(embeddings, cluster)

It is difficult to visualize the individual clusters due to the number of topics generated. However, we can see that even in 2-dimensional space some local structure is kept.

NOTE: You could skip the dimensionality reduction step if you use a clustering algorithm that can handle high dimensionality like a cosine-based k-Means.

## Topic Creation

What we want to know from the clusters that we generated, is what makes one cluster, based on their content, different from another?

How can we derive topics from clustered documents?

To solve this, I came up with a class-based variant of TF-IDF (c-TF-IDF), that would allow me to extract what makes each set of documents unique compared to the other.

The intuition behind the method is as follows. When you apply TF-IDF as usual on a set of documents, what you are basically doing is comparing the importance of words between documents.

What if, we instead treat all documents in a single category (e.g., a cluster) as a single document and then apply TF-IDF? The result would be a very long document per category and the resulting TF-IDF score would demonstrate the important words in a topic.

### c-TF-IDF
To create this class-based TF-IDF score, we need to first create a single document for each cluster of documents:

In [ ]:
# get docs_per_topic
docs_per_topic, docs_df = tf_idf(data, cluster)


Then, we apply the class-based TF-IDF by joining documents within a class. Image by the author.
Where the frequency of each word t is extracted for each class i and divided by the total number of words w. This action can be seen as a form of regularization of frequent words in the class. Next, the total, unjoined, number of documents m is divided by the total frequency of word t across all classes n.

In [ ]:
# get idf scores
tf_idf, count = c_tf_idf(docs_per_topic.Doc.values, m=len(data))


Now, we have a single importance value for each word in a cluster which can be used to create the topic. If we take the top 10 most important words in each cluster, then we would get a good representation of a cluster, and thereby a topic.

In order to create a topic representation, we take the top 20 words per topic based on their c-TF-IDF scores. The higher the score, the more representative it should be of its topic as the score is a proxy of information density.

In [ ]:
# create topic representation
top_n_words = extract_top_n_words_per_topic(tf_idf, count, docs_per_topic, n=20)
topic_sizes = extract_topic_sizes(docs_df); 


We can use topic_sizes to view how frequent certain topics are:

In [ ]:
print(topic_sizes.head(10))

The topic name-1 refers to all documents that did not have any topics assigned. The great thing about HDBSCAN is that not all documents are forced towards a certain cluster. If no cluster could be found, then it is simply an outlier.

We can see that topics 7,12 are the largest clusters that we could create. To view the words belonging to those topics, we can simply use the dictionarytop_n_words to access these topics:

In [ ]:
print(top_n_words[7][:5])

print(top_n_words[12][:5])

Looking at the largest four topics, I would say that these nicely seem to represent easily interpretable topics!

I can see sports, computers, space, and religion as clear topics that were extracted from the data.

## Topic Reduction

There is a chance that, depending on the dataset, you will get hundreds of topics that were created! You can tweak the parameters of HDBSCAN such that you will get fewer topics through its min_cluster_size parameter but it does not allow you to specify the exact number of clusters.

A nifty trick that Top2Vec was using is the ability to reduce the number of topics by merging the topic vectors that were most similar to each other.

We can use a similar technique by comparing the c-TF-IDF vectors among topics, merge the most similar ones, and finally re-calculate the c-TF-IDF vectors to update the representation of our topics:

In [ ]:
# topic reduction

docs_df_adj, top_n_words = topic_reduction(data, docs_df)
topic_sizes = extract_topic_sizes(docs_df_adj);

Above, we took the least common topic and merged it with the most similar topic. By repeating this 19 more times we reduced the number of topics from 56 to 36!

NOTE: We can skip the re-calculation part of this pipeline to speed up the topic reduction step. However, it is more accurate to re-calculate the c-TF-IDF vectors as that would better represent the newly generated content of the topics. You can play around with this by, for example, update every n steps to both speed-up the process and still have good topic representations.

TIP: You can use the method described in this article (or simply use BERTopic) to also create sentence-level embeddings. The main advantage of this is the possibility to view the distribution of topics within a single document.